In [ ]:
# Cargar datos

import sys
sys.path.append("..")     # para poder importar desde src/

from src.data import cargar_enemdu, indicadores_laborales
import pandas as pd

df = cargar_enemdu()
print(df.shape)
df.info(memory_usage="deep")
df.head()

(82894, 11)
<class 'pandas.DataFrame'>
RangeIndex: 82894 entries, 0 to 82893
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   area     82894 non-null  float64
 1   ciudad   82894 non-null  float64
 2   p02      82894 non-null  float64
 3   p03      82894 non-null  float64
 4   fexp     82894 non-null  float64
 5   nnivins  78757 non-null  float64
 6   ingrl    35870 non-null  float64
 7   condact  82894 non-null  float64
 8   secemp   39920 non-null  float64
 9   grupo1   39920 non-null  float64
 10  rama1    39920 non-null  float64
dtypes: float64(11)
memory usage: 7.0 MB


,area,ciudad,p02,p03,fexp,nnivins,ingrl,condact,secemp,grupo1,rama1
0,1.0,10150.0,2.0,69.0,46.2024,5.0,NaN,9.0,NaN,NaN,NaN
1,1.0,10150.0,1.0,28.0,46.2024,5.0,526.0,1.0,1.0,5.0,18.0
2,1.0,10150.0,2.0,19.0,46.2024,5.0,NaN,9.0,NaN,NaN,NaN
3,1.0,10150.0,2.0,87.0,46.2024,3.0,NaN,9.0,NaN,NaN,NaN
4,1.0,10150.0,2.0,65.0,46.2024,5.0,800.0,1.0,1.0,7.0,6.0


In [2]:
# Sanity check

print("Nulos por columna:")
print(df.isna().sum())

print("\nSuma de fexp (poblacion represent.):", f"{df['fexp'].sum():,.0f}")
print("\nDistribucion de condact (ponderada, en %):")
print((df.groupby("condact")["fexp"].sum() / df["fexp"].sum() * 100).round(2))

Nulos por columna:
area           0
ciudad         0
p02            0
p03            0
fexp           0
nnivins     4137
ingrl      47024
condact        0
secemp     42974
grupo1     42974
rama1      42974
dtype: int64

Suma de fexp (poblacion represent.): 19,026,779

Distribucion de condact (ponderada, en %):
condact
0.0    28.93
1.0    16.16
2.0     7.98
3.0     0.36
4.0    14.79
5.0     4.38
6.0     0.10
7.0     1.36
8.0     0.17
9.0    25.77
Name: fexp, dtype: float64


In [3]:
# Primera validacion

oficial = {"Participacion global": 63.8, "Empleo adecuado": 35.7,
            "Subempleo": 18.4, "Desempleo": 3.4}

calc = indicadores_laborales(df)
comparacion = pd.DataFrame({"Calculado": calc, "Oficial INEC": oficial}).round(1)
comparacion["Diferencia"] = (comparacion["Calculado"] - comparacion["Oficial INEC"]).round(1)
comparacion

,Calculado,Oficial INEC,Diferencia
Participacion global,63.7,63.8,-0.1
Empleo adecuado,35.7,35.7,0.0
Subempleo,18.4,18.4,0.0
Desempleo,3.4,3.4,0.0


In [6]:
# Variables que faltan

import pyreadstat
_, meta = pyreadstat.read_sav("../data/raw/spss/enemdu_persona_2026_l_trimestre.sav", metadataonly=True) 

for clave in ["prov", "esc", "anios", "horas", "p24", "p51", "inf", "etnia", "p15"]:
    hallazgos = [(c, meta.column_names_to_labels.get(c)) for c in meta.column_names if clave in c.lower()]
    print(clave, "->", hallazgos[:6])

prov -> []
esc -> []
anios -> []
horas -> []
p24 -> [('p24', 'Horas de trabajo en la semana anterior')]
p51 -> [('p51a', 'Horas de trabajo principal'), ('p51b', 'Horas de trabajo secundario'), ('p51c', 'Horas de otros trabajos')]
inf -> [('cod_inf', 'Código del Informante')]
etnia -> []
p15 -> [('p15', 'Como se considera'), ('p15aa', 'Dónde nació'), ('p15ab', 'Lugar de nacimiento')]


In [7]:
# Comprobacion diferencia 0.1 en participacion

c = df["condact"]
pea = c.between(1, 8)

pet_condact = c.between(1, 9)      # lo que usamos
pet_edad = df["p03"] >= 15         # alternativa por edad

for nombre, pet in [("PET por condact", pet_condact), ("PET por edad >= 15", pet_edad)]:
    tasa = 100 * df.loc[pea, "fexp"].sum() / df.loc[pet, "fexp"].sum()
    print(f"{nombre}: {tasa:.3f}")

PET por condact: 63.745
PET por edad >= 15: 63.745
